# Tool-calling agent with MCP servers

A platform team is replacing their custom tool-use framework with MCP because every new tool requires three weeks of integration work in the custom system. You have 90 minutes to build a Claude Sonnet 4.6 agent that wires up three MCP servers (a stock filesystem server, a stock HTTP fetch server, and a custom MCP server you write from scratch that wraps a SQLite internal database) and completes a multi-step task end-to-end: find the customer with the highest outstanding balance, draft an email to them, save the email to S3.

The killer moment: the lab's setup cell installs an interceptor that makes the database tool return `is_error=True` on its first call. A naive agent loop dies right there. A real production loop sees the error, asks Claude what to do, and Claude tries the same tool again. The trace shows two tool calls, the first failed, the second succeeded.

Estimated time: 90 minutes. Session window: 120 minutes. Cheapest lab in this sub-track. Most of the spend is Claude tokens; the MCP servers run inside your Colab kernel for free. A clean run is about 60 cents. A debugging run with five tool-error injections is around $1.20.

In [ ]:
# NBVAL_SKIP
# Pinned dependencies. mcp is the official MCP Python SDK; mcp-server-fetch
# and the stock filesystem server are exposed as console scripts the SDK
# spawns over stdio. psutil scans for orphan child processes at setup.

!pip install --quiet labrun-checks==0.7.0 anthropic==0.42.0 boto3==1.35.74 mcp==1.1.2 psutil==6.1.0

In [ ]:
# Imports and per-lab constants. The lab slug, S3 bucket name, and the two
# local file paths are pinned here so every later cell pulls from one place.

import asyncio
import atexit
import getpass
import json
import os
import re
import sqlite3
import subprocess
import sys
import time
from pathlib import Path

import boto3
import psutil
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupAdapter,
    CleanupResource,
)
from labrun_checks.adapters.aws import AwsCleanupAdapter

LAB_ID = "tool-calling-agent-with-mcp"
LAB_TAG_VALUE = LAB_ID
REGION = "us-east-1"

BUCKET_NAME = f"labrun-{LAB_ID}-bucket"
EMAIL_KEY = "email_draft.txt"

SQLITE_PATH = "/content/internal.db"
CUSTOM_SERVER_PATH = "/content/customer_mcp_server.py"

ANTHROPIC_MODEL = "claude-sonnet-4-6"
ANTHROPIC_HAIKU_MODEL = "claude-haiku-4-5-20250930"

# Tracked subprocess handles so cleanup can terminate them at session end.
MCP_PROCESSES: dict = {}

In [ ]:
# NBVAL_SKIP
# BYOK setup: session token, ANTHROPIC_API_KEY, AWS credentials. Ping Haiku
# to validate the Anthropic key cheaply, then STS GetCallerIdentity to confirm
# AWS, then register the session.

import anthropic

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
ANTHROPIC_API_KEY = getpass.getpass("ANTHROPIC_API_KEY: ").strip()
AWS_ACCESS_KEY_ID = getpass.getpass("AWS_ACCESS_KEY_ID: ").strip()
AWS_SECRET_ACCESS_KEY = getpass.getpass("AWS_SECRET_ACCESS_KEY: ").strip()

if not all([ANTHROPIC_API_KEY, AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY]):
    print("All three credentials are required. Re-run this cell.")
    raise SystemExit(1)

os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY
os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
os.environ["AWS_DEFAULT_REGION"] = REGION

# Cheap Anthropic ping. One Haiku call: a fraction of a cent.
try:
    _client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    _ping = _client.messages.create(
        model=ANTHROPIC_HAIKU_MODEL,
        max_tokens=8,
        messages=[{"role": "user", "content": "reply with the single word: ok"}],
    )
    print(f"Anthropic credentials validated. Haiku replied: {_ping.content[0].text!r}")
except Exception as exc:
    print(f"Anthropic key rejected: {exc!r}")
    print("Refresh ANTHROPIC_API_KEY and re-run this cell.")
    raise SystemExit(1)

# AWS STS check.
try:
    sts = boto3.client(
        "sts",
        region_name=REGION,
        aws_access_key_id=AWS_ACCESS_KEY_ID,
        aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    )
    ident = sts.get_caller_identity()
    ACCOUNT_ID = ident["Account"]
    print(f"AWS credentials validated. Account: {ACCOUNT_ID}. Region: {REGION}.")
except ClientError as exc:
    print(f"AWS credentials rejected: {exc!r}")
    print("Refresh AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY in your AWS console and re-run this cell.")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": AWS_ACCESS_KEY_ID,
    "aws_secret_access_key": AWS_SECRET_ACCESS_KEY,
    "region": REGION,
}

register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, custom AgentMcpCleanupAdapter that wraps AwsCleanupAdapter
# for S3 plus inline subprocess + local-file handling, atexit safety net,
# orphan scan for both S3 buckets and lingering MCP child processes.
#
# TODO: when labrun-checks ships a generic local_subprocess + local_file
# adapter, fold this inline class into that helper. Until then the lab
# carries the adapter so the build is self-contained.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="s3_object",
        id=EMAIL_KEY,
        parent=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rm s3://{BUCKET_NAME}/{EMAIL_KEY}",
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
    CleanupResource(
        type="local_subprocess",
        id="filesystem",
        region="local",
        cli_delete_command="pkill -f mcp-server-filesystem",
    ),
    CleanupResource(
        type="local_subprocess",
        id="fetch",
        region="local",
        cli_delete_command="pkill -f mcp-server-fetch",
    ),
    CleanupResource(
        type="local_subprocess",
        id="customer",
        region="local",
        cli_delete_command=f"pkill -f {CUSTOM_SERVER_PATH}",
    ),
    CleanupResource(
        type="local_file",
        id=CUSTOM_SERVER_PATH,
        region="local",
        cli_delete_command=f"rm -f {CUSTOM_SERVER_PATH}",
    ),
    CleanupResource(
        type="local_file",
        id=SQLITE_PATH,
        region="local",
        cli_delete_command=f"rm -f {SQLITE_PATH}",
    ),
]


class AgentMcpCleanupAdapter:
    """Wraps AwsCleanupAdapter for S3 and handles MCP subprocesses + local files inline.

    Delegates s3_bucket / s3_object to the shipped AWS adapter. Handles the
    two new types this lab introduces (local_subprocess, local_file) directly.
    TODO: migrate to a packaged local_subprocess adapter in labrun-checks 0.8.
    """

    def __init__(self, process_registry):
        self._aws = AwsCleanupAdapter()
        self._procs = process_registry

    def _terminate(self, process_key):
        proc = self._procs.get(process_key)
        if proc is None:
            return
        if proc.poll() is not None:
            return
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except subprocess.TimeoutExpired:
            proc.kill()
            try:
                proc.wait(timeout=2)
            except subprocess.TimeoutExpired:
                pass

    def delete_resource(self, credentials, resource):
        rtype = resource.type
        if rtype in ("s3_bucket", "s3_object"):
            self._aws.delete_resource(credentials, resource)
            return
        if rtype == "local_subprocess":
            self._terminate(resource.id)
            return
        if rtype == "local_file":
            try:
                os.remove(resource.id)
            except FileNotFoundError:
                pass
            return
        raise ValueError(f"AgentMcpCleanupAdapter: unknown resource type {rtype!r}")

    def describe_resource(self, credentials, resource):
        rtype = resource.type
        if rtype in ("s3_bucket", "s3_object"):
            return self._aws.describe_resource(credentials, resource)
        if rtype == "local_subprocess":
            proc = self._procs.get(resource.id)
            if proc is None:
                return False
            return proc.poll() is None
        if rtype == "local_file":
            return os.path.exists(resource.id)
        return False

    def tag_scan(self, credentials, lab_slug, region):
        if region == "local":
            # Look for stray python processes still running the custom server
            # by file path; the stock MCP servers do not have a stable path to
            # match against and are tracked via the process registry only.
            orphans = []
            try:
                for proc in psutil.process_iter(["pid", "cmdline"]):
                    cmdline = " ".join(proc.info.get("cmdline") or [])
                    if CUSTOM_SERVER_PATH in cmdline and proc.info["pid"] != os.getpid():
                        orphans.append(f"local:process:{proc.info['pid']}")
            except Exception:
                pass
            return orphans
        return self._aws.tag_scan(credentials, lab_slug, region)

    def exists(self, credentials, arn):
        return self._aws.exists(credentials, arn)


CLEANUP_ADAPTER = AgentMcpCleanupAdapter(process_registry=MCP_PROCESSES)


def _atexit_cleanup():
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


# Orphan scan, two surfaces. Bucket from a prior session would block create;
# stray python processes running the custom server file would leak Colab RAM.
def scan_for_orphans():
    s3_orphans = []
    try:
        s3 = boto3.client(
            "s3",
            region_name=REGION,
            aws_access_key_id=AWS_ACCESS_KEY_ID,
            aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
        )
        s3.head_bucket(Bucket=BUCKET_NAME)
        s3_orphans.append(BUCKET_NAME)
    except ClientError:
        pass
    proc_orphans = []
    try:
        for proc in psutil.process_iter(["pid", "cmdline"]):
            cmdline = " ".join(proc.info.get("cmdline") or [])
            if CUSTOM_SERVER_PATH in cmdline and proc.info["pid"] != os.getpid():
                proc_orphans.append(proc.info["pid"])
    except Exception:
        pass
    return s3_orphans, proc_orphans


s3_orphans, proc_orphans = scan_for_orphans()
if s3_orphans or proc_orphans:
    print("BLOCKED: leftover resources from a previous session were found.")
    for name in s3_orphans:
        print(f"  - S3 bucket: {name}")
    for pid in proc_orphans:
        print(f"  - Stray python process (likely custom MCP server): pid {pid}")
    print()
    print("Run the cleanup cell at the bottom first (it is idempotent and safe),")
    print("then re-run setup. If the python process orphans persist, restart the")
    print("Colab runtime: Runtime menu > Restart session.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

In [ ]:
# NBVAL_SKIP
# Seed the SQLite internal database the custom MCP server will wrap, and
# write the custom MCP server file to disk. The server is written from this
# notebook (not pip-installed) on purpose: the cognitive work in CP2 is
# building a tool schema and a SQL-backed handler.

# 1. SQLite seed. Six rows; the customer with the highest outstanding balance
#    is "Acme Industrial Co." with $48,210.00 unpaid. The agent should pick
#    this one.
if os.path.exists(SQLITE_PATH):
    os.remove(SQLITE_PATH)
conn = sqlite3.connect(SQLITE_PATH)
try:
    cur = conn.cursor()
    cur.execute(
        """
        CREATE TABLE customers (
            id INTEGER PRIMARY KEY,
            name TEXT NOT NULL,
            email TEXT NOT NULL,
            outstanding_balance REAL NOT NULL
        )
        """
    )
    cur.executemany(
        "INSERT INTO customers (name, email, outstanding_balance) VALUES (?, ?, ?)",
        [
            ("Riverbend Logistics", "ap@riverbendlogistics.example", 2150.50),
            ("Acme Industrial Co.", "finance@acme-industrial.example", 48210.00),
            ("Northpoint Studio", "billing@northpoint.example", 980.00),
            ("Goldleaf Bakery", "hello@goldleaf.example", 320.75),
            ("Summit Hardware", "accounts@summithardware.example", 12400.00),
            ("Bayview Press", "office@bayviewpress.example", 7625.40),
        ],
    )
    conn.commit()
finally:
    conn.close()

print(f"Seeded {SQLITE_PATH} with 6 customers.")

# 2. Custom MCP server: a small standalone python file that the mcp SDK will
#    spawn over stdio. Exposes ONE tool, `query_customers`, that accepts a
#    raw SQL SELECT string and returns the rows as JSON. A real production
#    server would validate the SQL; for this lab the constraint is the tool's
#    inputSchema (read-only SELECT) plus the SQLite connection being opened
#    read-only via URI flag.
CUSTOMER_SERVER_SOURCE = '''
import asyncio
import json
import sqlite3
import sys

from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp.types import Tool, TextContent

DB_PATH = "/content/internal.db"

server = Server("customer-db")


@server.list_tools()
async def list_tools():
    return [
        Tool(
            name="query_customers",
            description=(
                "Run a read-only SQL SELECT against the internal customers table. "
                "Columns: id, name, email, outstanding_balance. Use this to find "
                "customers by any condition and ordering."
            ),
            inputSchema={
                "type": "object",
                "properties": {
                    "sql": {
                        "type": "string",
                        "description": "A SELECT statement. Must begin with SELECT.",
                    }
                },
                "required": ["sql"],
            },
        )
    ]


@server.call_tool()
async def call_tool(name, arguments):
    if name != "query_customers":
        return [TextContent(type="text", text=json.dumps({"error": f"unknown tool {name}"}))]
    sql = (arguments or {}).get("sql", "").strip()
    if not sql.upper().startswith("SELECT"):
        return [TextContent(type="text", text=json.dumps({"error": "only SELECT statements allowed"}))]
    conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
    try:
        conn.row_factory = sqlite3.Row
        rows = [dict(r) for r in conn.execute(sql).fetchall()]
    finally:
        conn.close()
    return [TextContent(type="text", text=json.dumps(rows))]


async def main():
    async with stdio_server() as (read_stream, write_stream):
        await server.run(read_stream, write_stream, server.create_initialization_options())


if __name__ == "__main__":
    asyncio.run(main())
'''

Path(CUSTOM_SERVER_PATH).write_text(CUSTOMER_SERVER_SOURCE)
print(f"Wrote custom MCP server to {CUSTOM_SERVER_PATH}.")

In [ ]:
# NBVAL_SKIP
# Create the S3 bucket the agent will write the email draft into. us-east-1
# is the historical default region for CreateBucket and does NOT take a
# LocationConstraint; every other region does. The lab is pinned to us-east-1.

s3 = boto3.client(
    "s3",
    region_name=REGION,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
)
try:
    s3.create_bucket(Bucket=BUCKET_NAME)
    s3.get_waiter("bucket_exists").wait(Bucket=BUCKET_NAME)
    s3.put_bucket_tagging(
        Bucket=BUCKET_NAME,
        Tagging={"TagSet": [{"Key": "labrun:lab-slug", "Value": LAB_TAG_VALUE}]},
    )
    print(f"Created S3 bucket {BUCKET_NAME} in {REGION} and tagged it for orphan scans.")
except ClientError as exc:
    code = exc.response.get("Error", {}).get("Code")
    if code == "BucketAlreadyOwnedByYou":
        print(f"Bucket {BUCKET_NAME} already exists in this account. Continuing.")
    else:
        raise

## Task 1: Connect three MCP servers

Spawn three MCP servers as subprocesses and confirm each one responds to `list_tools` over stdio JSON-RPC.

1. The stock filesystem server (`mcp-server-filesystem`). Started with a single allowed root directory argument; expose `/content` so the agent can write a local file if it needs scratch space.
2. The stock HTTP fetch server (`mcp-server-fetch`). No arguments; lets the agent fetch URLs.
3. The custom customer-database server you wrote in the setup cell at `/content/customer_mcp_server.py`. Spawn it with `python /content/customer_mcp_server.py`.

For each, use the `mcp` Python SDK to open a `ClientSession` over the subprocess's stdio, call `list_tools()`, and confirm the returned list is non-empty. Track every subprocess Popen handle in `MCP_PROCESSES` so cleanup can terminate them at session end.

Pitfall: the MCP SDK manages the subprocess lifecycle when you use `stdio_client(StdioServerParameters(...))`, so passing the wrong command string makes the server look like it crashed when really it never started. The error surfaces as a timeout on `list_tools`, not a clear "command not found."


In [ ]:
# NBVAL_SKIP
# Task 1: spawn three MCP servers and call list_tools against each.

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

MCP_TOOLS_BY_SERVER: dict = {}


async def list_tools_for(server_key, params):
    """Open a stdio MCP session against `params`, return the tool list."""
    async with stdio_client(params) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            resp = await session.list_tools()
            tools = [
                {"name": t.name, "description": t.description}
                for t in resp.tools
            ]
            return tools


# YOUR CODE: build three StdioServerParameters objects, one per server.
# YOUR CODE: filesystem -> command="mcp-server-filesystem", args=["/content"]
# YOUR CODE: fetch      -> command="mcp-server-fetch", args=[]
# YOUR CODE: customer   -> command=sys.executable, args=[CUSTOM_SERVER_PATH]
# YOUR CODE: assign the three parameter objects to a dict named SERVER_PARAMS
# YOUR CODE: keyed by 'filesystem', 'fetch', 'customer'.

SERVER_PARAMS: dict = {}

# YOUR CODE: for each key in SERVER_PARAMS, await list_tools_for(key, params)
# YOUR CODE: and store the tool list under MCP_TOOLS_BY_SERVER[key]. Use
# YOUR CODE: asyncio.run on a small async wrapper, or call await directly if
# YOUR CODE: Colab's top-level await is enabled.

for key, tools in MCP_TOOLS_BY_SERVER.items():
    print(f"  {key}: {[t['name'] for t in tools]}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: all three servers responded to list_tools and returned at
# least one tool each. The validator re-runs list_tools (it does not trust
# whatever MCP_TOOLS_BY_SERVER captured) so a stale cache from a previous
# kernel does not pass.


def checkpoint_1(session):
    required = {"filesystem", "fetch", "customer"}
    if not isinstance(SERVER_PARAMS, dict) or set(SERVER_PARAMS.keys()) != required:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"SERVER_PARAMS must be a dict with keys exactly {sorted(required)}; "
                f"got {sorted(SERVER_PARAMS.keys()) if isinstance(SERVER_PARAMS, dict) else type(SERVER_PARAMS)!r}."
            ),
        )

    async def _probe():
        results = {}
        for key, params in SERVER_PARAMS.items():
            try:
                tools = await asyncio.wait_for(list_tools_for(key, params), timeout=20)
                results[key] = tools
            except Exception as exc:
                results[key] = f"ERROR: {exc!r}"
        return results

    probe = asyncio.run(_probe())
    for key in ("filesystem", "fetch", "customer"):
        tools = probe.get(key)
        if isinstance(tools, str) and tools.startswith("ERROR"):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Server {key!r} did not respond to list_tools. {tools}. The most common cause is"
                    f" a bad command string or args list on its StdioServerParameters."
                ),
            )
        if not tools:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Server {key!r} answered list_tools but returned zero tools. "
                    f"For the custom server, confirm the @server.list_tools() handler returns a non-empty list."
                ),
            )
    return CheckpointResult(status="pass")


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

MCP clients talk to servers over stdio JSON-RPC. The `mcp` SDK gives you `stdio_client(StdioServerParameters(command, args))` to spawn the subprocess and wire the pipes; inside that context manager, you open a `ClientSession`, call `initialize()`, then call `list_tools()`.

If a server is failing silently, the symptom looks like a timeout, not a clean error. Double-check the command name and the args list.

</details>

<details><summary>Hint 2 (direction)</summary>

Use `sys.executable` (not the literal string `"python"`) when spawning the custom server: Colab's kernel python is not always on PATH as `python`. For the stock servers, the pip install of `mcp` registers them as console scripts so the bare command name works.

```
SERVER_PARAMS = {
    "filesystem": StdioServerParameters(command="mcp-server-filesystem", args=["/content"]),
    "fetch": StdioServerParameters(command="mcp-server-fetch", args=[]),
    "customer": StdioServerParameters(command=sys.executable, args=[CUSTOM_SERVER_PATH]),
}
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Full working Task 1 body:

```python
SERVER_PARAMS = {
    "filesystem": StdioServerParameters(command="mcp-server-filesystem", args=["/content"]),
    "fetch": StdioServerParameters(command="mcp-server-fetch", args=[]),
    "customer": StdioServerParameters(command=sys.executable, args=[CUSTOM_SERVER_PATH]),
}

async def _probe_all():
    out = {}
    for key, params in SERVER_PARAMS.items():
        out[key] = await list_tools_for(key, params)
    return out

MCP_TOOLS_BY_SERVER = asyncio.run(_probe_all())

for key, tools in MCP_TOOLS_BY_SERVER.items():
    print(f"  {key}: {[t['name'] for t in tools]}")
```

</details>

**Wallet check.** No Claude calls yet; only Haiku ping ($0.0001) and three local subprocess starts (free). Total damage so far: less than a tenth of a cent.

## Task 2: Call the custom server's SQL tool

Open an MCP session against the custom customer-database server and call its `query_customers` tool with the SQL: `SELECT id, name, email, outstanding_balance FROM customers ORDER BY outstanding_balance DESC LIMIT 1`. Parse the JSON text content of the tool response and store the result row in a variable named `TOP_CUSTOMER`.

The point of this task is the round trip: your tool schema (`inputSchema` with one required string field `sql`) is the contract; your `@server.call_tool()` handler is the implementation; the MCP client serializes the call arguments to JSON, sends them over stdio, your handler runs the SQL, and the JSON-encoded rows come back as a `TextContent` payload.

If the call returns zero rows, the validator will fail with a clear message about the seed data.


In [ ]:
# NBVAL_SKIP
# Task 2: call query_customers and capture the top customer row.

TOP_CUSTOMER: dict = {}


async def call_customer_tool(sql_text):
    """Open an MCP session against the customer server and call query_customers."""
    params = SERVER_PARAMS["customer"]
    async with stdio_client(params) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            # YOUR CODE: call session.call_tool with name='query_customers' and
            # YOUR CODE: arguments={'sql': sql_text}. Capture the response.
            response = None
            # YOUR CODE: pull the first TextContent off response.content and
            # YOUR CODE: json.loads its .text attribute into a python list.
            # YOUR CODE: return that list.
            return []


# YOUR CODE: run call_customer_tool with the SELECT ... ORDER BY balance DESC LIMIT 1
# YOUR CODE: SQL, then set TOP_CUSTOMER to the first row of the returned list.

print("Top customer:", TOP_CUSTOMER)

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: the custom server exposes a tool that returns the seeded top
# customer row. Validator re-runs the SELECT against the live MCP server.


def checkpoint_2(session):
    async def _hit():
        params = SERVER_PARAMS["customer"]
        async with stdio_client(params) as (read_stream, write_stream):
            async with ClientSession(read_stream, write_stream) as s:
                await s.initialize()
                # Confirm the tool is declared.
                listed = await s.list_tools()
                tool_names = {t.name for t in listed.tools}
                if "query_customers" not in tool_names:
                    return None, f"custom server does not declare a 'query_customers' tool; got {sorted(tool_names)}"
                # Run the SELECT.
                resp = await s.call_tool(
                    "query_customers",
                    {"sql": "SELECT name, outstanding_balance FROM customers ORDER BY outstanding_balance DESC LIMIT 1"},
                )
                text = ""
                for part in resp.content:
                    if getattr(part, "text", None):
                        text = part.text
                        break
                try:
                    rows = json.loads(text)
                except Exception as exc:
                    return None, f"tool returned non-JSON text: {text!r}; parse error: {exc!r}"
                if not rows:
                    return None, "tool returned an empty result set; the seed data may have been wiped."
                return rows[0], None

    try:
        top, err = asyncio.run(_hit())
    except Exception as exc:
        return CheckpointResult(
            status="error",
            error_reason=f"could not call query_customers on the custom server: {exc!r}",
        )
    if err:
        return CheckpointResult(status="fail", error_reason=err)
    if top.get("name") != "Acme Industrial Co.":
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"query returned {top!r}; expected name='Acme Industrial Co.' as the top balance. "
                f"Confirm the SELECT orders by outstanding_balance DESC and limits to 1."
            ),
        )
    return CheckpointResult(status="pass")


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

`session.call_tool(name, arguments)` returns a `CallToolResult` with a `.content` list. Each entry is a `TextContent` (or `ImageContent` for image tools); pull the `.text` attribute and `json.loads` it. The custom server encodes the row list as a JSON string before sending.

</details>

<details><summary>Hint 2 (direction)</summary>

Two pitfalls hit students:

1. The handler decorator in the custom server is `@server.call_tool()` (with parens), not `@server.call_tool`. The decorator factory needs to be called.
2. `inputSchema` is JSON Schema, not Python type hints. Required fields go in the `required` array; the parameter shape lives in `properties`.

```
response = await session.call_tool("query_customers", {"sql": sql_text})
text = response.content[0].text
rows = json.loads(text)
return rows
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Full working Task 2 body:

```python
async def call_customer_tool(sql_text):
    params = SERVER_PARAMS["customer"]
    async with stdio_client(params) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            response = await session.call_tool(
                "query_customers", {"sql": sql_text}
            )
            text = next(p.text for p in response.content if getattr(p, "text", None))
            return json.loads(text)

rows = asyncio.run(
    call_customer_tool(
        "SELECT id, name, email, outstanding_balance "
        "FROM customers ORDER BY outstanding_balance DESC LIMIT 1"
    )
)
TOP_CUSTOMER = rows[0]
print("Top customer:", TOP_CUSTOMER)
```

</details>

**Wallet check.** Still no Claude calls. One Haiku ping (~$0.0001), one stdio tool call to a local subprocess (free). Total: less than a cent. Coffee remains 200x more expensive.

## Task 3: The full multi-step task

Wire Claude Sonnet 4.6 to all three MCP servers and have it complete this single instruction end-to-end:

> Find the customer with the highest outstanding balance, draft a polite payment-reminder email addressed to them, and save the email body as `email_draft.txt` to the S3 bucket `labrun-tool-calling-agent-with-mcp-bucket`. Include the customer's name in the email body. When you are done, reply with the string `TASK COMPLETE`.

The S3 PutObject is the agent's final side effect. The agent does NOT have a direct S3 tool; the lab does not expose an S3 MCP server (S3 is the validator surface, not a tool surface). The agent must use the filesystem MCP server to write a local file, then call back to your notebook code through an inline `s3_put_object` tool you expose in the tool catalog. That keeps the agent decision under Claude's control while the actual PutObject runs in this kernel.

Loop contract: while the latest assistant message contains any `tool_use` block, send a `user` message with one `tool_result` per `tool_use` (each with the tool's output text), then call `client.messages.create` again with the appended history. Stop when the latest assistant message contains no `tool_use` blocks. Cap at 25 iterations as a safety belt.


In [ ]:
# NBVAL_SKIP
# Task 3: connect Claude Sonnet 4.6 to the three MCP servers via an agent
# loop, plus an inline s3_put_object tool that this kernel handles directly.
# Trace every tool call so the error-recovery task in CP4 can read the history.

AGENT_TRACE: list = []
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

INSTRUCTION = (
    "Find the customer with the highest outstanding balance, "
    "draft a polite payment-reminder email addressed to them, "
    f"and save the email body as {EMAIL_KEY} to the S3 bucket {BUCKET_NAME}. "
    "Include the customer's name in the email body. "
    "When you are done, reply with the string TASK COMPLETE."
)


async def fetch_remote_mcp_tool_catalog():
    """Walk every MCP server, return a flat list of Anthropic tool specs.

    Each spec carries the MCP server key in name so we know which subprocess
    to dispatch a tool_use back to.
    """
    catalog = []
    for server_key, params in SERVER_PARAMS.items():
        async with stdio_client(params) as (read_stream, write_stream):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()
                listed = await session.list_tools()
                for t in listed.tools:
                    catalog.append({
                        "name": f"{server_key}__{t.name}",
                        "description": (t.description or "")[:500],
                        "input_schema": t.inputSchema or {"type": "object", "properties": {}},
                        "_mcp_server": server_key,
                        "_mcp_tool": t.name,
                    })
    return catalog


async def dispatch_mcp_tool(server_key, tool_name, tool_input):
    """Open a fresh MCP session to server_key and run tool_name(tool_input)."""
    params = SERVER_PARAMS[server_key]
    async with stdio_client(params) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            resp = await session.call_tool(tool_name, tool_input or {})
            text_parts = [getattr(p, "text", "") for p in resp.content if getattr(p, "text", None)]
            is_error = bool(getattr(resp, "isError", False))
            return "\n".join(text_parts), is_error


# Inline s3_put_object tool. Not from an MCP server; this kernel runs it
# directly and returns the result the same way an MCP tool_result would.
def dispatch_s3_put_object(tool_input):
    body = tool_input.get("body", "")
    key = tool_input.get("key", EMAIL_KEY)
    s3.put_object(Bucket=BUCKET_NAME, Key=key, Body=body.encode("utf-8"))
    return f"wrote s3://{BUCKET_NAME}/{key} ({len(body)} bytes)", False


S3_TOOL_SPEC = {
    "name": "s3_put_object",
    "description": (
        f"Write a UTF-8 text body to s3://{BUCKET_NAME}/<key>. Use this to "
        "save the final email draft."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "key": {"type": "string", "description": "object key under the bucket"},
            "body": {"type": "string", "description": "text payload to write"},
        },
        "required": ["key", "body"],
    },
}


def build_anthropic_tools(catalog):
    """Strip MCP routing fields before passing to anthropic.messages.create."""
    out = []
    for spec in catalog:
        out.append({
            "name": spec["name"],
            "description": spec["description"],
            "input_schema": spec["input_schema"],
        })
    out.append(S3_TOOL_SPEC)
    return out


def run_agent(instruction, catalog, max_iterations=25):
    """Agent loop. Loops while the latest assistant turn carries tool_use blocks.

    Returns (final_text, iterations, trace_length).
    """
    tools = build_anthropic_tools(catalog)
    name_to_route = {spec["name"]: (spec["_mcp_server"], spec["_mcp_tool"]) for spec in catalog}
    messages = [{"role": "user", "content": instruction}]

    final_text = ""
    for iteration in range(max_iterations):
        resp = client.messages.create(
            model=ANTHROPIC_MODEL,
            max_tokens=2048,
            tools=tools,
            messages=messages,
        )
        messages.append({"role": "assistant", "content": resp.content})

        tool_uses = [b for b in resp.content if b.type == "tool_use"]
        text_blocks = [b.text for b in resp.content if b.type == "text"]
        if text_blocks:
            final_text = text_blocks[-1]

        # YOUR CODE: if tool_uses is empty, break out of the loop. The assistant
        # YOUR CODE: has no more tool_use blocks, so it is done.

        # YOUR CODE: build a tool_result for each tool_use in tool_uses, then
        # YOUR CODE: append one user message carrying all tool_results.
        # YOUR CODE: For each tool_use:
        # YOUR CODE:   - if name == 's3_put_object', call dispatch_s3_put_object
        # YOUR CODE:   - else look up the (server, tool) in name_to_route and
        # YOUR CODE:     call asyncio.run(dispatch_mcp_tool(server, tool, input))
        # YOUR CODE: Each dispatch returns (text, is_error). Append a dict
        # YOUR CODE: with 'type': 'tool_result', 'tool_use_id': tu.id, 'content': text,
        # YOUR CODE: 'is_error': is_error to a list. Also append a dict with the
        # YOUR CODE: tool name, input, output, and is_error to AGENT_TRACE.
        # YOUR CODE: Then append {'role': 'user', 'content': tool_results} to messages.

        pass  # student replaces this

    return final_text, iteration + 1, len(AGENT_TRACE)


# Reset interceptor flags (CP4's interceptor lives in the next cell; we want
# CP3 to run cleanly without an injected error).
os.environ.pop("LABRUN_INJECT_TOOL_ERROR", None)
AGENT_TRACE.clear()
CATALOG = asyncio.run(fetch_remote_mcp_tool_catalog())
print(f"Tool catalog: {len(CATALOG)} MCP tools + 1 inline s3 tool.")

answer_text, iters, trace_len = run_agent(INSTRUCTION, CATALOG)
print(f"Agent finished in {iters} iterations with {trace_len} tool calls.")
print("Final text:", answer_text)

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: S3 HeadObject + GetObject + regex on the body. The validator
# hits the live bucket; it does not trust AGENT_TRACE.

EXPECTED_NAME = "Acme Industrial Co."


def checkpoint_3(session):
    try:
        s3.head_object(Bucket=BUCKET_NAME, Key=EMAIL_KEY)
    except ClientError as exc:
        code = exc.response.get("Error", {}).get("Code")
        if code in ("NoSuchKey", "NotFound", "404"):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"s3://{BUCKET_NAME}/{EMAIL_KEY} does not exist. The agent never called s3_put_object "
                    f"(or the tool was named differently in your catalog). Confirm the agent loop sends a "
                    f"user message with tool_result blocks after every assistant tool_use turn."
                ),
            )
        return CheckpointResult(status="error", error_reason=f"HeadObject failed: {exc!r}")
    obj = s3.get_object(Bucket=BUCKET_NAME, Key=EMAIL_KEY)
    body = obj["Body"].read().decode("utf-8", errors="replace")
    if not body.strip():
        return CheckpointResult(
            status="fail",
            error_reason="email_draft.txt exists but is empty; the agent passed an empty body to s3_put_object.",
        )
    if not re.search(re.escape(EXPECTED_NAME), body):
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"email body does not contain the expected customer name {EXPECTED_NAME!r}. "
                f"The agent picked the wrong customer or formatted the name differently. "
                f"Body starts with: {body[:120]!r}"
            ),
        )
    return CheckpointResult(status="pass")


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

The agent loop must keep calling `client.messages.create` while the latest assistant message has any `tool_use` block. The most common bug is breaking after the first tool call, which makes the agent look smart enough to read the database but never reach the S3 step.

Each `tool_use` needs exactly one matching `tool_result` (matched by `tool_use_id`). The `tool_result` blocks all go into a single user message after the assistant turn.

</details>

<details><summary>Hint 2 (direction)</summary>

Shape of the loop body once the assistant returns:

```
if not tool_uses:
    break
tool_results = []
for tu in tool_uses:
    text, is_error = dispatch(tu.name, tu.input)
    tool_results.append({
        "type": "tool_result",
        "tool_use_id": tu.id,
        "content": text,
        "is_error": is_error,
    })
    AGENT_TRACE.append({"tool": tu.name, "input": tu.input, "output": text, "is_error": is_error})
messages.append({"role": "user", "content": tool_results})
```

The dispatch should branch on `tu.name == "s3_put_object"` vs everything else.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Full working loop body (replace the `pass` line):

```python
if not tool_uses:
    break

tool_results = []
for tu in tool_uses:
    if tu.name == "s3_put_object":
        text, is_error = dispatch_s3_put_object(tu.input)
    else:
        server_key, real_tool = name_to_route[tu.name]
        text, is_error = asyncio.run(dispatch_mcp_tool(server_key, real_tool, tu.input))
    tool_results.append({
        "type": "tool_result",
        "tool_use_id": tu.id,
        "content": text,
        "is_error": is_error,
    })
    AGENT_TRACE.append({
        "tool": tu.name,
        "input": tu.input,
        "output": text,
        "is_error": is_error,
    })
messages.append({"role": "user", "content": tool_results})
```

</details>

**Wallet check.** Multi-step task done. The agent made roughly 4-6 tool calls and 5-7 LLM round trips at Sonnet 4.6 rates. Spent: ~$0.40 on Claude. S3 PutObject + bucket: free tier. Cumulative session: about 40 cents.

## Task 4: Recover from an injected tool error

Set the env var `LABRUN_INJECT_TOOL_ERROR=1` and re-run the agent loop. The interceptor below wraps `dispatch_mcp_tool` so the FIRST call to `customer__query_customers` returns `is_error=True` with a `500: database temporarily unavailable` body. Every subsequent call succeeds normally.

A naive agent loop dies right there (it ignores `is_error` or it treats the error string as the answer). A real production loop sees `is_error=True` in the `tool_result` it just appended and re-prompts: the next iteration of the loop sends the error back to Claude, Claude reads it, retries the same tool, and the second call lands clean.

Your loop from Task 3 already supports this because Anthropic's API expects you to pass the `is_error` flag straight through; Claude reads the tool_result text and decides on its own to retry. Nothing changes in the loop body. The only thing you do here is set the env var, clear the trace, and re-invoke.

Validator: `AGENT_TRACE` for this run shows at least two `customer__query_customers` entries; the first has `is_error=True`, the second has `is_error=False`.


In [ ]:
# NBVAL_SKIP
# Task 4: rerun the agent loop with a one-shot is_error interceptor wrapping
# the customer__query_customers tool. The interceptor flips is_error=True on
# the FIRST call to that tool; all subsequent calls return cleanly.

_QUERY_ERROR_FIRED = {"yes": False}
_real_dispatch_mcp_tool = dispatch_mcp_tool


async def dispatch_mcp_tool_with_interceptor(server_key, tool_name, tool_input):
    if os.environ.get("LABRUN_INJECT_TOOL_ERROR") == "1":
        if server_key == "customer" and tool_name == "query_customers" and not _QUERY_ERROR_FIRED["yes"]:
            _QUERY_ERROR_FIRED["yes"] = True
            return "500: database temporarily unavailable. retry recommended.", True
    return await _real_dispatch_mcp_tool(server_key, tool_name, tool_input)


dispatch_mcp_tool = dispatch_mcp_tool_with_interceptor

# YOUR CODE: set os.environ["LABRUN_INJECT_TOOL_ERROR"] = "1"
# YOUR CODE: clear AGENT_TRACE
# YOUR CODE: reset the _QUERY_ERROR_FIRED flag to {"yes": False}
# YOUR CODE: delete the previous email_draft.txt from S3 so the validator
# YOUR CODE: confirms the agent recreated it from a fresh run
# YOUR CODE: call run_agent(INSTRUCTION, CATALOG); print the iterations and trace length

recovery_text = ""
recovery_iters = 0

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: AGENT_TRACE shows two query_customers calls, the first failed,
# the second succeeded. The email_draft.txt exists with the right name.


def checkpoint_4(session):
    customer_calls = [
        entry for entry in AGENT_TRACE
        if entry.get("tool") == "customer__query_customers"
    ]
    if len(customer_calls) < 2:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"AGENT_TRACE shows only {len(customer_calls)} call(s) to customer__query_customers. "
                f"Expected at least 2 (the first injected as 500, the second succeeding). "
                f"Either LABRUN_INJECT_TOOL_ERROR was not set, the AGENT_TRACE was not cleared before "
                f"the recovery run, or the agent loop broke after the first error instead of looping."
            ),
        )
    if not customer_calls[0].get("is_error"):
        return CheckpointResult(
            status="fail",
            error_reason=(
                "The first customer__query_customers call in AGENT_TRACE did not record is_error=True. "
                "The interceptor flag was probably already consumed by a stale run; reset "
                "_QUERY_ERROR_FIRED['yes'] = False and clear AGENT_TRACE before re-running."
            ),
        )
    later_success = any(not c.get("is_error") for c in customer_calls[1:])
    if not later_success:
        return CheckpointResult(
            status="fail",
            error_reason=(
                "Every call to customer__query_customers in AGENT_TRACE recorded is_error=True. "
                "The agent never got a successful database read after the first failure. "
                "Confirm the loop continues iterating after appending an is_error tool_result."
            ),
        )
    try:
        s3.head_object(Bucket=BUCKET_NAME, Key=EMAIL_KEY)
    except ClientError as exc:
        return CheckpointResult(
            status="fail",
            error_reason=f"After recovery, email_draft.txt is missing from the bucket. HeadObject error: {exc!r}",
        )
    return CheckpointResult(status="pass")


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

Anthropic does not auto-retry tool errors. The retry only happens because your loop body keeps iterating; the `is_error=True` field on a `tool_result` is just text Claude can read on the next turn.

The reason your CP3 loop already supports retry is that you pass `is_error` straight through to the API. If you hardcoded `is_error: false`, Claude has no signal that the tool actually failed.

</details>

<details><summary>Hint 2 (direction)</summary>

Three things to reset before the recovery run:

1. `os.environ["LABRUN_INJECT_TOOL_ERROR"] = "1"` so the interceptor is armed.
2. `AGENT_TRACE.clear()` so the validator only sees this run's calls, not Task 3's.
3. `_QUERY_ERROR_FIRED["yes"] = False` so the interceptor fires once on this run (not zero times because Task 3 already used it).

Also delete the prior `email_draft.txt` so the validator confirms the recovery run wrote a fresh one.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Full working Task 4 trigger block:

```python
os.environ["LABRUN_INJECT_TOOL_ERROR"] = "1"
AGENT_TRACE.clear()
_QUERY_ERROR_FIRED["yes"] = False

try:
    s3.delete_object(Bucket=BUCKET_NAME, Key=EMAIL_KEY)
except ClientError:
    pass

recovery_text, recovery_iters, _ = run_agent(INSTRUCTION, CATALOG)
print(f"Recovery run: {recovery_iters} iterations, {len(AGENT_TRACE)} tool calls.")
print("Final text:", recovery_text)
os.environ.pop("LABRUN_INJECT_TOOL_ERROR", None)
```

</details>

**Wallet check.** Recovery run added 1-2 extra Claude turns vs Task 3 because Claude had to read the 500 error and decide to retry. Roughly $0.15 more. Cumulative: about 55 cents. Cheapest agent lab on labrun.

## Cleanup

Three-phase teardown via `run_cleanup`: empty the bucket and delete the email object, delete the bucket, terminate every MCP subprocess with a 5-second grace before SIGKILL, delete `/content/customer_mcp_server.py` and `/content/internal.db`. The verification scan re-queries S3 HeadBucket and re-checks each Popen handle and local file; a stale resource triggers `sys.exit(1)`.


In [ ]:
# NBVAL_SKIP
# Cleanup. Idempotent: re-running this cell is safe.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges (S3 is pennies, but resolve anyway).")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: about $0.60.** One Haiku ping plus one clean multi-step run plus one error-recovery run on Claude Sonnet 4.6. Roughly 25K tokens. S3 was free tier. The three MCP servers ran inside your Colab kernel for nothing.

MCP servers killed. Bucket emptied and deleted. SQLite file gone. Custom server file gone. The Colab runtime is back to a clean slate.

## Reflection

These are not graded. They are for you.

1. The agent retried once after the database tool returned 500. In production you cannot retry forever; a permanently broken tool would lock the agent in a retry loop until the iteration cap (or the token budget) fires. What is a sensible per-task retry budget for tool errors, and how would you wire it into the agent loop without re-prompting Claude every time? Walk through where the counter lives (per tool name, per agent task, or per session) and what the agent does when the budget is exhausted (skip the tool, fail the task, or fall back to a different tool).

2. You built one custom MCP server in 90 minutes. The platform team says they have 47 internal tools to wrap (databases, internal HTTP APIs, message queues, feature flags, ticketing systems). What is the most useful abstraction you would build before writing the 4th server? Schema generation from existing SDKs, a base `Server` subclass with shared auth and tracing, a tool registry pattern, or something else? Be specific about which engineering decisions the abstraction freezes.

## Exam-Style Practice

**Question 1 (MCP architecture):**

A team has 30 internal tools they want to expose to a Claude agent. They are deciding between (a) wrapping each tool as a separate MCP server, (b) putting all 30 tools inside one MCP server, or (c) skipping MCP and registering 30 tools directly in the Anthropic Messages API `tools` parameter. The team needs hot-swappable tools (the agent should be able to call a tool that was added last week without redeploying), versioned tool schemas, and the ability to use the same tools from different LLMs (Claude, GPT, local models). Which approach matches all three requirements?

A. Wrap each tool as a separate MCP server.
B. Put all 30 tools inside one MCP server.
C. Skip MCP and register the 30 tools directly in the Anthropic `tools` parameter.
D. Use a custom non-MCP protocol that the team controls end-to-end.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: 30 separate servers gives you 30 subprocesses, 30 deploy targets, and 30 places to update auth. Hot-swap is theoretically possible but you have paid an enormous operational tax. This is over-engineering for the requirements.
- B is correct: one MCP server with all 30 tools registered under it lets you add a new tool by editing one repo, redeploy once, and every MCP-aware LLM (Claude, GPT via an MCP bridge, local models with MCP support) gets the tool the moment they reconnect. The protocol is the abstraction that makes the tools portable across LLMs.
- C is wrong: registering tools directly in `tools` couples them to one provider's API shape, which fails the cross-LLM requirement. There is no schema versioning surface either.
- D is wrong: a custom protocol gives you the same coupling problem as C plus the additional cost of building the protocol.

</details>

**Question 2 (tool error handling):**

An agent uses three MCP servers. One returns a 5xx persistently for every call. The agent should:

A. Retry forever until the server recovers.
B. Circuit-break after N failures and continue the task without that tool.
C. Crash the agent loop and surface the failure to the caller.
D. Swap to a different LLM model and retry the call.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: retry-forever locks the agent loop. The token budget runs out, the user waits forever, and the offending server is not going to recover just because you ask it more times.
- B is correct: bounded retry plus graceful degradation is the production pattern. Track failures per tool; after N consecutive 5xx, mark the tool unavailable in the catalog for this task, and let the agent attempt the task with the remaining tools (or fail the task with a clear error if that tool was essential).
- C is wrong: crashing on first persistent failure throws away progress from every other tool. A task might be 80% done; the user does not want a full restart because tool 3 of 5 is down.
- D is wrong: the failure is on the tool side, not the model side. Swapping LLMs does nothing for a 5xx server.

</details>

**Question 3 (tool schema design):**

A custom MCP server exposes a `query_customers` tool that takes a SQL string. The team reviews the design and flags it as risky. Which mitigation does the LEAST while still letting the agent ask varied questions?

A. Replace the `sql` parameter with a fixed set of structured filters (e.g., `min_balance`, `sort_by`, `limit`).
B. Keep the `sql` parameter but constrain the tool to read-only operations (SELECT only) and run the connection in read-only mode.
C. Remove the tool entirely; have the agent ask the user to write the SQL.
D. Hardcode one specific SELECT statement; the tool takes no arguments.

<details><summary>Show answer</summary>

**Correct: B.**

- A is correct in spirit (most secure) but does the MOST: structured filters mean every new query shape needs a code change, which kills the flexibility advantage of giving the agent SQL.
- B is correct: SELECT-only validation in the handler plus opening the SQLite/Postgres connection in read-only mode gives the agent room to compose any read query while preventing writes or schema changes. This is what the lab's custom server does.
- C is wrong: removing the tool defeats the point of having an agent.
- D is wrong: hardcoding one query means the agent can only ever ask one question; you have built a function call dressed up as a tool.

</details>